In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as sp
import qutip as qt
from tqdm.auto import tqdm

In [ ]:
# System Parameters for Single Symbol Simulation
alpha = 2.0             # Coherent state amplitude (|alpha_0> = |+alpha>, |alpha_1> = |-alpha>)
alpha0 = alpha          # Hypothesis 0 amplitude
alpha1 = -alpha         # Hypothesis 1 amplitude
nu = 2.0                # Dark count rate (counts per second)
T = 1.0                 # Symbol duration (seconds)
dt = 1e-4               # Time step resolution
prior_P0 = 0.5          # Initial prior probability for state |alpha_0>
steps = int(T / dt)


In [ ]:
true_state_idx = 1
true_alpha = alpha0 if true_state_idx == 0 else alpha1

# Initialize tracking variables
P0 = prior_P0
P0_history = np.zeros(steps)
beta_history = np.zeros(steps)
clicks = []

# Continuous time-slice simulation of the optimal Dolinar receiver with off-center displacement
for i in tqdm(range(steps), desc="Simulating Single Symbol"):
    t = i * dt
    P0_history[i] = P0
    
    # 1. Optimal Dolinar off-center displacement field beta(t)
    # Time-to-go offset factor: f(t) = exp(-4 * alpha^2 * (T - t))
    # beta(t) = -alpha * [ (2*P0 - 1) + sqrt(1 - f(t)) ] / [ 1 + (2*P0 - 1)*sqrt(1 - f(t)) ]
    f_t = np.exp(-4.0 * alpha**2 * (T - t))
    denom = np.sqrt(np.maximum(1.0 - f_t, 1e-12))
    x = 2.0 * P0 - 1.0
    beta = -alpha * (x + denom) / (1.0 + x * denom)
    beta_history[i] = beta
    
    # 2. Instantaneous Poisson click rates (displaced signal + dark counts)
    rate0 = np.abs(alpha0 + beta)**2 + nu
    rate1 = np.abs(alpha1 + beta)**2 + nu
    rate_true = np.abs(true_alpha + beta)**2 + nu
    
    # 3. Exact on/off detector click probability in interval dt
    p_click_true = 1.0 - np.exp(-rate_true * dt)
    
    # 4. Bayesian continuous drift and discrete jump state update
    if np.random.rand() < p_click_true:
        # Detector clicked: discrete jump update
        num = P0 * rate0
        den = num + (1.0 - P0) * rate1
        P0 = num / den if den > 0 else 0.5
        clicks.append(t)
    else:
        # No click: continuous deterministic drift update
        num = P0 * np.exp(-rate0 * dt)
        den = num + (1.0 - P0) * np.exp(-rate1 * dt)
        P0 = num / den if den > 0 else 0.5
        
    P0 = np.clip(P0, 1e-12, 1.0 - 1e-12)

# Final decision based on ending posterior probability
decision = 0 if P0 > 0.5 else 1
print(f"Completed: True state = |alpha_{true_state_idx}>, Receiver Decision = |alpha_{decision}>, Total Clicks = {len(clicks)}")


In [ ]:
time_axis = np.linspace(0, T, steps)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Plot Posterior Probability P0(t)
ax1.plot(time_axis, P0_history, label=r'Posterior Probability $P_0(t)$', color='darkmagenta', linewidth=1.8)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Decision Boundary (0.5)')
for click in clicks:
    ax1.axvline(x=click, color='red', linestyle=':', alpha=0.8)
if clicks:
    ax1.axvline(x=clicks[0], color='red', linestyle=':', alpha=0.8, label='Detector Click')
ax1.set_ylabel(r"$P_0(t)$", fontsize=11)
ax1.set_title(f"Optimal Dolinar Receiver: True State $|\\alpha_{true_state_idx}\\rangle$, Decision $|\\alpha_{decision}\\rangle$ (Dark counts $\\nu={nu}$)")
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot Dynamic Off-Center Displacement Field beta(t)
ax2.plot(time_axis, beta_history, label=r'Off-Center LO Displacement $\beta(t)$', color='teal', linewidth=1.8)
ax2.axhline(-alpha, color='blue', linestyle=':', alpha=0.5, label=r'Fixed Kennedy Nulling ($-\alpha$)')
ax2.axhline(alpha, color='orange', linestyle=':', alpha=0.5, label=r'Fixed Kennedy Nulling ($+\alpha$)')
for click in clicks:
    ax2.axvline(x=click, color='red', linestyle=':', alpha=0.8)
ax2.set_xlabel("Time (s)", fontsize=11)
ax2.set_ylabel(r"$\beta(t)$", fontsize=11)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Theoretical Quantum Bounds for BPSK with Arbitrary Prior Probability
def helstrom_bound(alpha, pi0=0.5):
    """Quantum Helstrom minimum error probability bound for BPSK with prior pi0 = P(H0)"""
    pi1 = 1.0 - pi0
    overlap_sq = np.exp(-4.0 * alpha**2)
    return 0.5 * (1.0 - np.sqrt(np.maximum(1.0 - 4.0 * pi0 * pi1 * overlap_sq, 0.0)))

def homodyne_bound(alpha, pi0=0.5):
    """Standard Quantum Limit (SQL) via ideal homodyne detection with MAP threshold for prior pi0"""
    pi1 = 1.0 - pi0
    x_th = (1.0 / (4.0 * np.maximum(alpha, 1e-6))) * np.log(pi1 / pi0)
    err0 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha - x_th))
    err1 = 0.5 * sp.erfc(np.sqrt(2.0) * (alpha + x_th))
    return pi0 * err0 + pi1 * err1

def kennedy_bound(alpha, pi0=0.5):
    """Kennedy receiver theoretical error probability bound (fixed nulling of state 0)"""
    pi1 = 1.0 - pi0
    return pi1 * np.exp(-4.0 * alpha**2)

# 1. Simulation of the Kennedy Receiver (Static Displacement, NO Feedback)
def simulate_kennedy(alpha, prior_p0=0.5, nu=0.0, eta=1.0, T=1.0, trials=5000, disp_scale=1.0):
    """
    Simulates the static Kennedy receiver with a fixed displacement beta = -disp_scale * alpha.
    No dynamic feedback: count total photons in [0, T] and decide 0 if 0 clicks, else 1.
    """
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    
    # Static displacement (default disp_scale = 1.0 -> beta = -alpha)
    beta = -disp_scale * alpha
    
    # Total photon arrival rate at detector
    rate_true = eta * np.abs(true_alphas + beta)**2 + nu
    
    # Total photon count in interval T drawn from Poisson distribution
    clicks = np.random.poisson(rate_true * T)
    
    # Standard Kennedy decision: 0 clicks -> 0, >= 1 clicks -> 1
    decisions = np.where(clicks == 0, 0, 1)
    
    return np.mean(decisions != true_states)

# 2. Simulation of the Dolinar Receiver (Dynamic Continuous Feedback)
def simulate_event_driven_dolinar(alpha, prior_p0=0.5, nu=0.0, eta=1.0, latency=0.0, T=1.0, trials=5000):
    """
    Simulates the active Dolinar receiver with real-time Bayesian continuous feedback.
    """
    true_states = np.random.choice([0, 1], p=[prior_p0, 1.0 - prior_p0], size=trials)
    true_alphas = np.where(true_states == 0, alpha, -alpha)
    decisions = np.zeros(trials, dtype=int)
    
    for i in range(trials):
        s_alpha = true_alphas[i]
        t = 0.0
        P0 = prior_p0
        
        while t < T:
            f_t = np.exp(-4.0 * alpha**2 * (T - t))
            denom = np.sqrt(max(1.0 - f_t, 1e-12))
            x = 2.0 * P0 - 1.0
            beta = -alpha * (x + denom) / (1.0 + x * denom)
            
            rate_signal = np.abs(s_alpha + beta)**2
            rate_total = eta * rate_signal + nu
            
            if rate_total <= 1e-12:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
                
            delta_t = np.random.exponential(1.0 / rate_total)
            t_next = t + delta_t
            
            if t_next >= T:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                dt_rem = T - t
                P0 = P0 * np.exp(-rate0 * dt_rem) / (P0 * np.exp(-rate0 * dt_rem) + (1.0 - P0) * np.exp(-rate1 * dt_rem))
                break
            else:
                rate0 = np.abs(alpha + beta)**2 + nu
                rate1 = np.abs(-alpha + beta)**2 + nu
                P0 = P0 * np.exp(-rate0 * delta_t) / (P0 * np.exp(-rate0 * delta_t) + (1.0 - P0) * np.exp(-rate1 * delta_t))
                P0 = (P0 * rate0) / (P0 * rate0 + (1.0 - P0) * rate1)
                
                t = t_next + latency
                P0 = np.clip(P0, 1e-12, 1.0 - 1e-12)
                
        decisions[i] = 0 if P0 > 0.5 else 1
        
    return np.mean(decisions != true_states)

# Sweep functions
def sweep_ber(alpha_vals, sim_fn, **kwargs):
    desc = kwargs.pop('desc', 'Simulating')
    bers = []
    for a in tqdm(alpha_vals, desc=desc):
        bers.append(sim_fn(a, **kwargs))
    return np.array(bers)

# Configuration Parameters
prior_prob = 0.5          # Prior probability P(H0) (e.g. 0.5 for equal prior)
alpha_points = np.linspace(0.1, 1.6, 16)
trials_count = 5000

# Run Simulations for both Dolinar (Feedback) and Kennedy (No Feedback)
ber_dolinar_ideal = sweep_ber(alpha_points, simulate_event_driven_dolinar, prior_p0=prior_prob, nu=0.0, eta=1.0, trials=trials_count, desc=f"Dolinar (Ideal, P0={prior_prob})")
ber_dolinar_dark = sweep_ber(alpha_points, simulate_event_driven_dolinar, prior_p0=prior_prob, nu=0.5, eta=1.0, trials=trials_count, desc=f"Dolinar (Dark Counts nu=0.5, P0={prior_prob})")

ber_kennedy_ideal = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, nu=0.0, eta=1.0, trials=trials_count, desc=f"Kennedy (Ideal, P0={prior_prob})")
ber_kennedy_dark = sweep_ber(alpha_points, simulate_kennedy, prior_p0=prior_prob, nu=0.5, eta=1.0, trials=trials_count, desc=f"Kennedy (Dark Counts nu=0.5, P0={prior_prob})")

# Smooth theoretical curves
alpha_grid = np.linspace(0.05, 1.8, 200)

plt.figure(figsize=(11, 7))
plt.semilogy(alpha_grid, helstrom_bound(alpha_grid, pi0=prior_prob), 'k-', linewidth=2.5, label=f'Helstrom Bound ($P_0={prior_prob}$)')
plt.semilogy(alpha_grid, homodyne_bound(alpha_grid, pi0=prior_prob), 'g--', linewidth=2.0, label=f'Homodyne Detection (SQL, $P_0={prior_prob}$)')
plt.semilogy(alpha_grid, kennedy_bound(alpha_grid, pi0=prior_prob), 'c-.', linewidth=2.0, label=f'Kennedy Bound ($P_0={prior_prob}$)')

# Plot Simulated Dolinar (Feedback) points
plt.semilogy(alpha_points, ber_dolinar_ideal, 'mo', markersize=7, label=f'Simulated Dolinar (Ideal, $\\nu=0$)')
plt.semilogy(alpha_points, ber_dolinar_dark, 'rs', markersize=6, label=f'Simulated Dolinar (Dark Counts $\\nu=0.5$)')

# Plot Simulated Kennedy (No Feedback) points
plt.semilogy(alpha_points, ber_kennedy_ideal, 'c+', markersize=8, markeredgewidth=2, label=f'Simulated Kennedy (Ideal, $\\nu=0$)')
plt.semilogy(alpha_points, ber_kennedy_dark, 'kx', markersize=8, markeredgewidth=2, label=f'Simulated Kennedy (Dark Counts $\\nu=0.5$)')

plt.title(f'BPSK Detection: Dolinar (Feedback) vs. Kennedy (Static Displacement) vs. Quantum Bounds ($P_0={prior_prob}$)', fontsize=12)
plt.xlabel(r'Coherent State Amplitude $\alpha$ (Mean Photon Number $\bar{n}=\alpha^2$)', fontsize=11)
plt.ylabel('Bit Error Rate (BER)', fontsize=11)
plt.ylim(1e-6, 0.8)
plt.xlim(0.1, 1.8)
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.legend(loc='best', fontsize=9.5)
plt.show()